# 03 — Entrenamiento y Evaluación: YOLOv8n vs YOLOv8s
**Proyecto:** Placas-Deteccion-Robo
**Puntos del PDF:** 5) Entrenamiento y Evaluación de Modelos, 6) Selección y Exportación del Modelo Final

Continúa desde `02_preprocessing.ipynb` (mismo Colab, ya existe `data/processed/data.yaml`).

Comparación (Opción A del plan, la más simple): **YOLOv8n** (nano) vs **YOLOv8s** (small), fine-tuned pocas épocas (20–30) sobre el dataset chico de placas.


In [5]:
import os

# Si corre en Colab: monta Google Drive y se para en la carpeta ml-service
# ahi adentro, para que 01/02/03 compartan data/ y model/ entre runtimes
# distintos. Si corre local (VS Code), solo ajusta el cwd a ml-service/.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    # Ajusta esta ruta si pusiste la carpeta en otro lugar de tu Drive
    DRIVE_PATH = '/content/drive/MyDrive/Placas-Deteccion-Robo/ml-service'
    os.chdir(DRIVE_PATH)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print('Working directory:', os.getcwd())


Working directory: c:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service


### 3.1 Instalación

In [2]:
!pip install -q ultralytics


In [6]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('ADVERTENCIA: va a entrenar en CPU, va a ser mucho mas lento.')


CUDA disponible: True
GPU: NVIDIA GeForce GTX 1650


### 3.2 Entrenamiento — YOLOv8n

In [8]:
from ultralytics import YOLO

In [ ]:
model_n = YOLO('yolov8n.pt')  # preentrenado en COCO, se hace fine-tuning

results_n = model_n.train(
    data='data/processed/data.yaml',
    epochs=30,            # antes 5 (muy pocas para converger, ver hallazgo en revisión)
    imgsz=640,
    batch=8,               # antes 16: en GPU local de 4GB VRAM (ej. GTX 1650) evita
                           # quedarse sin memoria a imgsz=640. En Colab (GPU con más
                           # VRAM) se puede volver a subir a 16 si querés acelerar.
    name='placas_yolov8n',
    patience=10,           # early stopping: corta si no mejora en 10 épocas seguidas
    # --- augmentation extra: el dataset son fotos "de catálogo" (autos bien
    # encuadrados, buena luz); en producción la imagen viene de una cámara de
    # calle/celular con ángulos y distancias que el dataset no cubre. Estos
    # parámetros simulan esas condiciones para generalizar mejor: ---
    degrees=10.0,          # antes 0.0: tolera placas levemente rotadas
    perspective=0.0005,    # antes 0.0: simula ángulo de cámara no frontal (rango recomendado 0-0.001)
    mixup=0.15,            # antes 0.0: regularización extra (dataset chico, ~300 imgs train)
)


New https://pypi.org/project/ultralytics/8.4.107 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.10.5 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolov8n.pt, data=data/processed/data.yaml, epochs=30, time=None, patience=10, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=placas_yolov8n, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=Fa

train: Scanning C:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\data\processed\labels\train.cache... 303 images, 0 backgrounds, 0 corrupt: 100%|██████████| 303/303 [00:00<?, ?it/s]
val: Scanning C:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\data\processed\labels\val.cache... 86 images, 0 backgrounds, 0 corrupt: 100%|██████████| 86/86 [00:00<?, ?it/s]


Plotting labels to runs\detect\placas_yolov8n\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 63 weight(decay=0.0), 70 weight(decay=0.0005), 69 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs\detect\placas_yolov8n
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/30      1.19G      1.566      3.466      1.397         12        640: 100%|██████████| 38/38 [00:28<00:00,  1.36it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:08<00:00,  1.39s/it]

                   all         86         93    0.00353      0.978      0.523      0.201


...

      25/30       1.2G      1.252     0.9741      1.164          7        640: 100%|██████████| 38/38 [00:15<00:00,  2.42it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  4.63it/s]

                   all         86         93      0.953      0.867      0.949       0.51
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 15, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



25 epochs completed in 0.137 hours.
Optimizer stripped from runs\detect\placas_yolov8n\weights\last.pt, 5.6MB
Optimizer stripped from runs\detect\placas_yolov8n\weights\best.pt, 5.6MB

Validating runs\detect\placas_yolov8n\weights\best.pt...
Ultralytics 8.3.0  Python-3.10.5 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:01<00:00,  3.31it/s]


                   all         86         93      0.895      0.828      0.896      0.517
Speed: 0.3ms preprocess, 14.3ms inference, 0.0ms loss, 1.6ms postprocess per image
Results saved to runs\detect\placas_yolov8n


### 3.3 Entrenamiento — YOLOv8s

In [9]:
model_s = YOLO('yolov8s.pt')

results_s = model_s.train(
    data='data/processed/data.yaml',
    epochs=30,             # antes 5 (mismo ajuste que YOLOv8n, ver celda anterior)
    imgsz=640,
    batch=8,               # antes 16: YOLOv8s pesa más que el nano, con 4GB de VRAM
                           # local batch=16 probablemente tira CUDA out of memory.
    name='placas_yolov8s',
    patience=10,
    degrees=10.0,
    perspective=0.0005,
    mixup=0.15,
)


New https://pypi.org/project/ultralytics/8.4.107 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.10.5 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: task=detect, mode=train, model=yolov8s.pt, data=data/processed/data.yaml, epochs=30, time=None, patience=10, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=placas_yolov8s4, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=F

train: Scanning C:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\data\processed\labels\train.cache... 303 images, 0 backgrounds, 0 corrupt: 100%|██████████| 303/303 [00:00<?, ?it/s]
val: Scanning C:\KAPA-2025\VIco\Placas-Deteccion-Robo\ml-service\data\processed\labels\val.cache... 86 images, 0 backgrounds, 0 corrupt: 100%|██████████| 86/86 [00:00<?, ?it/s]


Plotting labels to runs\detect\placas_yolov8s4\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 63 weight(decay=0.0), 70 weight(decay=0.0005), 69 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs\detect\placas_yolov8s4
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/30      2.23G      1.576      2.887      1.369         12        640: 100%|██████████| 38/38 [00:51<00:00,  1.35s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:08<00:00,  1.50s/it]

                   all         86         93    0.00333      0.925      0.478      0.219


...

      30/30      2.18G      1.206     0.7875      1.162          7        640: 100%|██████████| 38/38 [00:40<00:00,  1.06s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:03<00:00,  1.94it/s]

                   all         86         93      0.893      0.901      0.931      0.515



30 epochs completed in 0.385 hours.
Optimizer stripped from runs\detect\placas_yolov8s4\weights\last.pt, 19.9MB
Optimizer stripped from runs\detect\placas_yolov8s4\weights\best.pt, 19.9MB

Validating runs\detect\placas_yolov8s4\weights\best.pt...
Ultralytics 8.3.0  Python-3.10.5 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
Model summary (fused): 186 layers, 9,828,051 parameters, 0 gradients, 23.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 6/6 [00:03<00:00,  1.75it/s]


                   all         86         93      0.924      0.914      0.943      0.522
Speed: 0.6ms preprocess, 32.8ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to runs\detect\placas_yolov8s4


### 3.4 Evaluación sobre el set de validación (mAP, precisión, recall)

In [1]:
#metrics_n = model_n.val(data='data/processed/data.yaml', split='val')
metrics_s = model_s.val(data='data/processed/data.yaml', split='val')

import pandas as pd

comparacion = pd.DataFrame({
    'modelo': ['YOLOv8n', 'YOLOv8s'],
    'mAP50': [metrics_n.box.map50, metrics_s.box.map50],
    'mAP50-95': [metrics_n.box.map, metrics_s.box.map],
    'precision': [metrics_n.box.mp, metrics_s.box.mp],
    'recall': [metrics_n.box.mr, metrics_s.box.mr],
})
comparacion


NameError: name 'model_s' is not defined

### 3.5 Comparación de velocidad de inferencia (relevante para elegir el modelo final)

In [ ]:
import time
import glob
import cv2

test_images = glob.glob('data/processed/images/test/*')[:20]

def bench(model, images):
    # warm-up
    _ = model.predict(images[0], verbose=False)
    t0 = time.time()
    for img in images:
        _ = model.predict(img, verbose=False)
    return (time.time() - t0) / len(images)

t_n = bench(model_n, test_images)
t_s = bench(model_s, test_images)

print(f'YOLOv8n: {t_n*1000:.1f} ms/imagen')
print(f'YOLOv8s: {t_s*1000:.1f} ms/imagen')


YOLOv8n: 68.7 ms/imagen
YOLOv8s: 122.4 ms/imagen


### 3.6 Selección del modelo final

Criterio de selección: mejor balance entre **mAP@0.5** y **velocidad de inferencia**, priorizando velocidad porque el servicio (`FastAPI /detect`) responde a pedidos síncronos desde el frontend — no hay tiempo para modelos pesados.

**Resultados de la corrida con 30 épocas + augmentation (degrees, perspective, mixup):**

| Modelo | Precision | Recall | mAP50 | mAP50-95 |
|---|---|---|---|---|
| YOLOv8n (`runs/detect/placas_yolov8n`) | 0.953 | 0.867 | **0.949** | 0.510 |
| YOLOv8s (`runs/detect/placas_yolov8s4`) | 0.893 | 0.901 | 0.931 | 0.515 |

**Decisión: YOLOv8n.** No hubo ni que aplicar el criterio de desempate por velocidad —
YOLOv8n queda arriba en mAP50 (0.949 vs 0.931) y en precision, además de ser ~4x más
liviano y más rápido de inferencia. YOLOv8s gana apenas en recall y mAP50-95, por
márgenes chicos que no justifican el costo extra de latencia para este servicio.

Comparado con la corrida original de 5 épocas (mAP50 0.479, recall 0.129), este
modelo pasa de no detectar casi nada a un desempeño sólido — confirmado también
de forma visual e in-vivo contra el ml-service real: 95.5% de las imágenes del test
set (42/44) con bbox detectado, confianza promedio 0.771 (antes: 1/16 en una
inspección visual manual, con falsos positivos).


### 3.7 Exportación del modelo elegido a ONNX (para el punto 6 y la integración FastAPI del punto 8)

In [ ]:
# Ganador de 3.6: YOLOv8n
best_model = model_n

export_path = best_model.export(format='onnx')
print('Modelo exportado en:', export_path)

# Guarda también el .pt entrenado (útil si se prefiere correr con ultralytics
# directo en FastAPI en vez de ONNX). OJO: ajustar 'placas_yolov8n' al nombre
# real de la carpeta en runs/detect/ que generó ESTA corrida (Ultralytics le
# agrega un sufijo numérico si el nombre ya existía, ej. placas_yolov8n2/3/...).
import shutil
shutil.copy('runs/detect/placas_yolov8n/weights/best.pt', 'model/yolov8n_placas.pt')
shutil.copy('runs/detect/placas_yolov8n/weights/best.onnx', 'model/yolov8n_placas.onnx')


### 3.8 Descargar los pesos para subirlos a `ml-service/model/` del proyecto

En Colab:
```python
from google.colab import files
files.download('model/yolov8n_placas.pt')
files.download('runs/detect/placas_yolov8n/weights/best.onnx')
```

Copiar ambos archivos a `ml-service/model/` en el repo local antes de levantar FastAPI.
